# Dave timing accuracy — actual vs. estimated

`before_imaging/regular/04_create_dave_config.ipynb` prints an *estimated* run
time per Dave block (imaging round / fluidics step), based on HAL exposure
times and Kilroy protocol durations (`MERci.acquisition.dave.estimate_dave_experiment`).
That estimate is fixed at generation time and never sees how the real
acquisition is actually going.

This notebook rebuilds the same block-by-block breakdown from the *real*
Dave recipe files this experiment is running (`SAMPLE_DIR/settings/dave-*.xml`),
then, for every block:
- **already happened** (its imaging round is fully written, or -- for a
  fluidics step -- the round before it is fully written and the round after
  it has started): fills in the *actual* start/end/duration, read from the
  mtime of the first/last expected image file (`MERci.common.io.path_mtime`,
  the same "when did writing finish" convention `ExperimentStateMonitor`
  already uses for imaging/idle detection).
- **not yet happened**: estimates its duration from Dave's own number,
  scaled by the average `actual / Dave` ratio observed so far across every
  *other* completed block of the same kind (imaging or fluidics), then
  chains it onto the previous block's end time -- so every future block gets
  an updated ETA instead of Dave's original static one.

A block whose imaging round has started but isn't fully written yet gets a
real observed *start* (first file already on disk) with an *estimated* end
(same ratio-scaled duration) -- flagged `estimated` in the status column,
since its end time isn't real yet.

Completed blocks' actual times are cached under
`analysis/cache/dave_timing_accuracy/` so a long acquisition's already-
finished rounds are never re-scanned.

**Re-run every cell any time to refresh** -- one-shot style like
`stage_z_drift.ipynb`, not a live-loop like `round_mosaics.ipynb`/
`imaged_fovs.ipynb`; add a loop later if watching this update continuously
turns out to be worth it.

The second table (section 8) is a coarser Dave-vs-actual sanity check for
1 frame / 1 FOV / all FOVs of one imaging round, plus one fluidics step --
each read off the most recently completed hyb round / fluidics block (falls
back to the cells round if no hyb round is done yet).

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
from IPython.display import display

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import path_mtime
from MERci.acquisition.dave       import (
    dave_cells_config_filename, estimate_dave_experiment, get_hal_frame_count,
)
from MERci.acquisition.kilroy     import find_kilroy_config
from MERci.analysis.stage_z       import round_label

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
MICROSCOPE = "MF3"   # EDIT ME -- must match this experiment's real microscope

SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "dave_timing_accuracy"   # used to namespace this notebook's cache files

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# NOTEBOOK_GUIDELINES.md #2: cache under analysis/cache/<notebook_name>/.
CACHE_DIR  = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / "block_timing.csv"

print(f"SAMPLE_NAME: {SAMPLE_NAME}")
print(f"Rounds in round_info.csv: {meta.valid_round_ids()}")

## 3 — Locate this experiment's Dave recipes and reproduce Dave's own estimate

Resolves the exact cells/hybs recipe files `04_create_dave_config.ipynb` wrote
for this experiment (`dave_cells_config_filename`/`dave_config_filename`'s own
naming convention), then reparses them with the same
`estimate_dave_experiment` that notebook used, so "Dave's estimate" here is
never re-derived by hand -- it's the identical number that notebook printed.

In [ ]:
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")

cells_recipe = config.settings_dir / dave_cells_config_filename(MICROSCOPE, SAMPLE_NAME)
if not cells_recipe.exists():
    raise FileNotFoundError(f"Cells Dave recipe not found: {cells_recipe}")

# The hybs recipe's filename encodes N_HYBS (dave_config_filename), which this
# notebook has no independent source for -- glob instead, requiring exactly one
# match (dave_config_filename's own docstring flags globbing as ambiguous when
# several dave-*hybs-*.xml exist in the same settings/; this experiment should
# only ever have one).
hybs_candidates = sorted(config.settings_dir.glob(f"dave-{MICROSCOPE.lower()}-*hybs-{SAMPLE_NAME}.xml"))
if len(hybs_candidates) != 1:
    raise FileNotFoundError(
        f"Expected exactly one hybs Dave recipe matching "
        f"'dave-{MICROSCOPE.lower()}-*hybs-{SAMPLE_NAME}.xml' in {config.settings_dir}, "
        f"found {len(hybs_candidates)}: {hybs_candidates}"
    )
hybs_recipe = hybs_candidates[0]

cells_est = estimate_dave_experiment(cells_recipe, kilroy_config=KILROY_CONFIG, settings_dir=config.settings_dir)
hybs_est  = estimate_dave_experiment(hybs_recipe,  kilroy_config=KILROY_CONFIG, settings_dir=config.settings_dir)

print(f"Cells recipe: {cells_recipe.name}")
print(f"Hybs recipe : {hybs_recipe.name}")
if cells_est.warnings or hybs_est.warnings:
    print("Warnings:", *cells_est.warnings, *hybs_est.warnings, sep="\n  ")

## 4 — Round ↔ hyb-number lookup, and the ordered block list

`round_label` (from `analysis/stage_z.py`) is the same round-id → `"cells"`/
`"hybNN"` mapping `stage_z_drift.ipynb` already uses, reused here rather than
re-deriving it, so this notebook's hyb numbering can never drift from that
one's.

`estimate_dave_experiment`'s own `per_round` merges a hyb's fluidics + imaging
loops into one dict (both loops share the same hyb-number key) -- split back
out into separate rows here, one per Dave *action*, since that's the
granularity actually requested (a fluidics line and an imaging line, not one
merged row per hyb).

In [ ]:
round_ids_sorted = meta.valid_round_ids()
round_labels     = {rid: round_label(meta, rid) for rid in round_ids_sorted}
cells_round_id   = next(rid for rid, lbl in round_labels.items() if lbl == "cells")
hyb_round_id     = {int(lbl[3:]): rid for rid, lbl in round_labels.items() if lbl.startswith("hyb")}

blocks = []
for r in cells_est.per_round:   # exactly one row: "Cells Imaging"
    blocks.append({"block": r["label"], "kind": "imaging", "hyb": None,
                    "round_id": cells_round_id, "dave_s": r["imaging_s"]})
for r in hybs_est.per_round:
    if r["hyb"] is not None:
        rid = hyb_round_id[r["hyb"]]
        blocks.append({"block": f"{r['label']} Fluidics", "kind": "fluidics", "hyb": r["hyb"],
                        "round_id": rid, "dave_s": r["fluidics_s"]})
        blocks.append({"block": f"{r['label']} Imaging", "kind": "imaging", "hyb": r["hyb"],
                        "round_id": rid, "dave_s": r["imaging_s"]})
    else:
        # "Fluidics Final" (or a segment-mode variant) -- no imaging round follows
        # it in this recipe, so it has no observable end anchor (section 5) and
        # always shows Dave's own estimate.
        blocks.append({"block": r["label"], "kind": "fluidics", "hyb": None,
                        "round_id": None, "dave_s": r["fluidics_s"]})

print(f"{len(blocks)} Dave blocks, in run order:")
for b in blocks:
    print(f"  {b['block']:<20s} ({b['kind']:<9s})  dave estimate: {b['dave_s']:.0f} s")

## 5 — Measure actual start/end per block (cached once final)

A block's actual start/end is read from the mtime of the first/last expected
image file of its round(s) (`common.io.path_mtime`). An imaging block is
*final* once its round is fully written (`ExperimentMetadata.round_fully_written`);
a fluidics block is *final* once the round before it is fully written AND the
round after it has at least one file on disk (that file's own mtime never
changes once written, even before its round finishes). Final blocks are
cached -- only genuinely new ones are (re)measured on each run.

In [ ]:
def _existing_files(round_id):
    if round_id is None:
        return []
    return [f for f in meta.files_for_round(round_id) if f.exists()]


def _round_span(round_id):
    """(min_mtime, max_mtime) across round_id's currently-existing expected
    files, or (None, None) if none exist yet. Tolerates a .zarr store
    directory that HAL has created but not yet written any chunk into
    (path_mtime raises FileNotFoundError on an empty directory store) --
    skipped, picked up on a later run once real data lands in it."""
    mtimes = []
    for f in _existing_files(round_id):
        try:
            mtimes.append(path_mtime(f))
        except FileNotFoundError:
            pass
    if not mtimes:
        return None, None
    return min(mtimes), max(mtimes)


if CACHE_PATH.exists():
    cached = pd.read_csv(CACHE_PATH).set_index("block").to_dict("index")
else:
    cached = {}

newly_measured = 0
for b in blocks:
    if b["block"] in cached:
        b["obs_start"] = cached[b["block"]]["obs_start"]
        b["obs_end"]   = cached[b["block"]]["obs_end"]
        b["measured"]  = True
        continue

    if b["kind"] == "imaging":
        done   = meta.round_fully_written(b["round_id"])
        lo, hi = _round_span(b["round_id"])
        if done and lo is not None:
            b["obs_start"], b["obs_end"], b["measured"] = lo, hi, True
        elif lo is not None:
            b["obs_start"], b["obs_end"], b["measured"] = lo, None, False    # in progress
        else:
            b["obs_start"], b["obs_end"], b["measured"] = None, None, False  # not started

    else:  # fluidics
        if b["round_id"] is None:   # "Fluidics Final" -- no next round to bound it
            b["obs_start"], b["obs_end"], b["measured"] = None, None, False
        else:
            prev_round_id       = cells_round_id if b["hyb"] == 1 else hyb_round_id[b["hyb"] - 1]
            prev_done            = meta.round_fully_written(prev_round_id)
            _, prev_end          = _round_span(prev_round_id)
            next_start, _        = _round_span(b["round_id"])
            if prev_done and prev_end is not None and next_start is not None:
                b["obs_start"], b["obs_end"], b["measured"] = prev_end, next_start, True
            elif prev_done and prev_end is not None:
                b["obs_start"], b["obs_end"], b["measured"] = prev_end, None, False   # fluidics running now
            else:
                b["obs_start"], b["obs_end"], b["measured"] = None, None, False

    if b["measured"]:
        newly_measured += 1

final_rows = [
    {"block": b["block"], "obs_start": b["obs_start"], "obs_end": b["obs_end"]}
    for b in blocks if b["measured"]
]
if final_rows:
    pd.DataFrame(final_rows).to_csv(CACHE_PATH, index=False)

print(f"{sum(b['measured'] for b in blocks)}/{len(blocks)} blocks fully measured "
      f"({newly_measured} newly this run).")

## 6 — Fill in estimates for not-yet-finished blocks, chained onto real progress

For each kind (imaging / fluidics), `mean(actual_s / dave_s)` over every
fully-measured block of that kind so far -- falling back to `1.0` (just
Dave's own number) if nothing of that kind has finished yet. Every block is
then walked in run order: a finished block keeps its real times; an
in-progress block keeps its real start but gets a ratio-scaled end; a
not-yet-started block gets both start and end chained off the previous
block's end.

In [ ]:
def _ratio_for(kind):
    ratios = [
        (b["obs_end"] - b["obs_start"]) / b["dave_s"]
        for b in blocks
        if b["measured"] and b["kind"] == kind and b["dave_s"] > 0
    ]
    return sum(ratios) / len(ratios) if ratios else 1.0


imaging_ratio  = _ratio_for("imaging")
fluidics_ratio = _ratio_for("fluidics")
print(f"Observed/Dave ratio so far -- imaging: {imaging_ratio:.2f}x, fluidics: {fluidics_ratio:.2f}x")

cursor = datetime.now()
rows = []
for b in blocks:
    ratio          = imaging_ratio if b["kind"] == "imaging" else fluidics_ratio
    est_duration_s = b["dave_s"] * ratio

    if b["measured"]:
        start_dt = datetime.fromtimestamp(b["obs_start"])
        end_dt   = datetime.fromtimestamp(b["obs_end"])
    elif b["obs_start"] is not None:   # in progress: real start, estimated end
        start_dt = datetime.fromtimestamp(b["obs_start"])
        end_dt   = start_dt + timedelta(seconds=est_duration_s)
    else:                              # not started yet: chain off the previous block
        start_dt = cursor
        end_dt   = start_dt + timedelta(seconds=est_duration_s)
    cursor = end_dt

    rows.append({
        "block":    b["block"],
        "kind":     b["kind"],
        "start":    start_dt,
        "end":      end_dt,
        "duration": end_dt - start_dt,
        "status":   "measured" if b["measured"] else "estimated",
    })

timing_table = pd.DataFrame(rows)

## 7 — Block-by-block timing table

Re-run sections 5-7 any time to refresh with whatever has imaged since the last run.

In [ ]:
display_df = timing_table.copy()
display_df["start"]    = display_df["start"].dt.strftime("%Y-%m-%d %H:%M:%S")
display_df["end"]      = display_df["end"].dt.strftime("%Y-%m-%d %H:%M:%S")
display_df["duration"] = display_df["duration"].apply(lambda d: str(d).split(".")[0])
display(display_df[["block", "kind", "start", "end", "duration", "status"]])

print(f"\nProjected experiment end: {timing_table['end'].iloc[-1]:%Y-%m-%d %H:%M:%S}")

## 8 — Dave vs. actual: 1 frame / 1 FOV / all FOVs / fluidics stage

Reference imaging round: the most recently completed **hyb** round (hyb
rounds dominate total experiment time and repeat identically, so they're the
more representative choice), falling back to the cells round if no hyb round
is done yet. Reference fluidics block: the most recently completed fluidics
step. Each row is blank until its own reference block has completed at least
once.

In [ ]:
done_hyb_blocks = [b for b in blocks if b["kind"] == "imaging" and b["hyb"] is not None and b["measured"]]
if done_hyb_blocks:
    ref_imaging = done_hyb_blocks[-1]
else:
    cells_block = next(b for b in blocks if b["kind"] == "imaging" and b["hyb"] is None)
    ref_imaging = cells_block if cells_block["measured"] else None

done_fluidics_blocks = [b for b in blocks if b["kind"] == "fluidics" and b["measured"]]
ref_fluidics = done_fluidics_blocks[-1] if done_fluidics_blocks else None

table2_rows = []

if ref_imaging is not None:
    n_fovs = len(meta.fovs)
    series = next((s for s in meta.series_for_round(ref_imaging["round_id"]) if s.hal_config), None)
    n_frames = get_hal_frame_count(config.settings_dir / series.hal_config) if series is not None else None

    dave_all_fovs_s   = ref_imaging["dave_s"]
    actual_all_fovs_s = ref_imaging["obs_end"] - ref_imaging["obs_start"]
    dave_fov_s        = dave_all_fovs_s / n_fovs
    actual_fov_s      = actual_all_fovs_s / n_fovs

    table2_rows.append({"metric": "all FOVs (1 round)", "dave_estimate_s": dave_all_fovs_s,
                         "actual_s": actual_all_fovs_s, "based_on": f"{ref_imaging['block']} ({n_fovs} FOVs)"})
    table2_rows.append({"metric": "1 FOV", "dave_estimate_s": dave_fov_s,
                         "actual_s": actual_fov_s, "based_on": ref_imaging["block"]})
    if n_frames:
        table2_rows.append({"metric": "1 frame", "dave_estimate_s": dave_fov_s / n_frames,
                             "actual_s": actual_fov_s / n_frames,
                             "based_on": f"{ref_imaging['block']} ({n_frames} frames/FOV)"})
else:
    for metric in ("all FOVs (1 round)", "1 FOV", "1 frame"):
        table2_rows.append({"metric": metric, "dave_estimate_s": None, "actual_s": None,
                             "based_on": "no round completed yet"})

if ref_fluidics is not None:
    table2_rows.append({"metric": "fluidics stage", "dave_estimate_s": ref_fluidics["dave_s"],
                         "actual_s": ref_fluidics["obs_end"] - ref_fluidics["obs_start"],
                         "based_on": ref_fluidics["block"]})
else:
    table2_rows.append({"metric": "fluidics stage", "dave_estimate_s": None, "actual_s": None,
                         "based_on": "no fluidics step completed yet"})

table2 = pd.DataFrame(table2_rows)

table2_display = table2.copy()
for col in ("dave_estimate_s", "actual_s"):
    table2_display[col] = table2_display[col].apply(
        lambda s: (str(timedelta(seconds=round(s))) if pd.notna(s) else "—"))
display(table2_display)